[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kithhooni-commits/ds-practice/blob/main/%EC%8B%A4%EC%8A%B55/colab_day3_v2.ipynb)

# 3일차 v2 — 1일차 답 + 2일차 답을 한 모델로

**다른 노트북과 독립적으로 돈다.** 처음부터 끝까지 이것만 실행하면 된다.

## 왜 이 구조인가

전달 곡선을 재보니 3일차가 정확히 분해된다 (val 30장).

| 측정치 영역에서 지운 정도 | 최종 PSNR | 최종 SSIM | 최적 K |
|---|---|---|---|
| 25 dB | 20.24 | 0.5572 | 1e-2 |
| 35 dB | 26.83 | 0.7809 | 3.2e-3 |
| **40 dB** | **29.86** | **0.8626** | 1.8e-3 |
| 45 dB | 32.70 | 0.9066 | 5.6e-4 |
| 완벽 (오차 0) | **71.47** | 0.9999 | 1e-8 |

마지막 줄이 핵심이다. **구조적 한계가 없다.** 측정치에서 노이즈만 완벽히 지우면
2일차 답이 그대로 71 dB 를 낸다. 최종 점수는 오직 디노이징 품질로 결정된다.

노이즈는 흐림 **뒤에** 붙었으므로 측정치 위에서는 백색이다 — **1일차 문제 그대로**다.

    z = net(g)             1일차 답 (DRUNet 37.42 를 warm start)
    x = (D·Z)/(D² + λ)     2일차 답 (역필터). 학습 파라미터는 λ 뿐

손실은 최종 이미지에서 잰다. 그래야 채점 SSIM 을 손실에 넣을 수 있고 λ 도 같이
학습된다 — 위 표에서 최적 K 가 네 자릿수를 움직이므로 고정값으로는 못 맞춘다.

## 통과 기준

**PSNR >= 26, SSIM >= 0.83.** SSIM 이 진짜 관문이다.


## 0. 런타임 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 1. 데이터

Drive 에 올린 dataset zip 을 푼다. 이미 풀려 있으면 건너뛴다.

In [ ]:
import os, zipfile, glob
from pathlib import Path

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(exist_ok=True)

if not (DATA_ROOT / "test_deconv_noise").exists():
    zips = sorted(glob.glob("/content/drive/MyDrive/**/dataset*.zip", recursive=True))
    print("찾은 zip:", *zips, sep="\n  ")
    for z in zips[-1:]:
        with zipfile.ZipFile(z) as f:
            f.extractall(DATA_ROOT)
    inner = DATA_ROOT / "dataset"
    if inner.exists() and not (DATA_ROOT / "test_deconv_noise").exists():
        DATA_ROOT = inner

for d in ("train", "val", "test_label", "test_deconv_noise"):
    n = len(glob.glob(str(DATA_ROOT / d / "*.npy")))
    print(f"  {d:<22}{n:>6}장")
assert (DATA_ROOT / "test_deconv_noise" / "noise_meta.json").exists(), "noise_meta.json 이 없다"
print(f"\nDATA_ROOT = {DATA_ROOT}")

## 2. 코드

In [ ]:
REPO = Path("/content/ds-practice")
if REPO.exists():
    !cd "{REPO}" && git pull --ff-only
else:
    !git clone https://github.com/kithhooni-commits/ds-practice.git "{REPO}"

SRC = REPO / "실습5" / "src" / "deconv"
RUNS = Path("/content/runs"); RUNS.mkdir(exist_ok=True)
!cd "{REPO}" && git log --oneline -1

## 3. 1일차 DRUNet 을 찾는다

측정치 영역 디노이저의 출발점으로 쓴다. 없어도 돌아가지만 있으면 훨씬 빨리 수렴한다.
σ 채널은 0 으로 채워 넣으므로 **시작 시점엔 1일차 디노이저와 정확히 같게** 동작한다.

In [ ]:
import torch

CAND = sorted(set(
    glob.glob("/content/drive/MyDrive/**/*.ckpt", recursive=True)
    + glob.glob("/content/runs/**/*.ckpt", recursive=True)))
D1, best = None, -1
for c in CAND:
    try:
        ck = torch.load(c, map_location="cpu", weights_only=False)
    except Exception:
        continue
    if ck.get("model") == "drunet" and not ck.get("label_free"):
        v = ck.get("val_psnr", 0)
        print(f"  후보  {v:6.2f} dB   {c}")
        if v > best:
            D1, best = c, v

INIT = f'--init-refine "{D1}"' if D1 else ""
print("\n선택:", D1 if D1 else "없음 — 무작위 초기화로 진행 (그래도 돌아간다)")

## 4. 넘어야 할 선

배포 baseline 과 학습 없는 조합들. K 는 전부 **val 에서** 고른다.

In [ ]:
!cd "{SRC}" && python run_day3.py --data "{DATA_ROOT}" --n-val 60

## 5. 주력 — twostage

**이것 하나만 돌려도 된다.** 나머지는 여유가 있을 때다.

`--batch 8` 에서 OOM 이 나면 4 로 낮출 것. `--lr-spectral` 은 주파수별 λ 전용 학습률이다. **로그 공간 파라미터라
2일차의 큰 값을 그대로 쓰면 안 된다** — Adam 은 스텝당 대략 lr 만큼 움직이므로
lr 0.05 면 한 에폭(921스텝)에 λ 가 e^37 배 튄다. 1e-3 ~ 3e-3 이 적당하다.

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model twostage --refine drunet --features 64 \
    --sigma-map --lam-map --lr-spectral 2e-3 {INIT} \
    --loss charbonnier_ssim --ssim-weight 0.5 \
    --noise-model challenge --input measure \
    --epochs 80 --batch 8 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag twostage

### 5b. SSIM 을 더 세게 — 0.83 에 못 미치면

SSIM 항의 비중을 0.84 로 올린다 (Zhao et al. 권장값). PSNR 을 조금 내주고
SSIM 을 사는 거래다. 위 결과가 SSIM 0.83 을 넘겼으면 **돌릴 필요 없다.**

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model twostage --refine drunet --features 64 \
    --sigma-map --lam-map --lr-spectral 2e-3 {INIT} \
    --loss charbonnier_ssim --ssim-weight 0.84 \
    --noise-model challenge --input measure \
    --epochs 80 --batch 8 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag twostage_s084

### 5c. 역필터 뒤에 한 번 더 다듬기

측정치 영역에서 못 지운 것이 역산으로 증폭돼 남으면 그것만 정리한다.
용량이 2배라 느리다. 융합 재료로도 쓴다.

In [ ]:
!cd "{SRC}" && python train_deconv.py \
    --model twostage --refine drunet --features 48 --refine-iters 1 \
    --sigma-map --lam-map --lr-spectral 2e-3 {INIT} \
    --loss charbonnier_ssim --ssim-weight 0.5 \
    --noise-model challenge --input measure \
    --epochs 80 --batch 4 --lr 2e-4 --clip-grad 1.0 --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag twostage_refine

## 6. 평가

`--self-ensemble` 은 4x 다. dipole 이 견디는 대칭만 쓴다 (좌우·상하·180도).
90도 회전은 B0 방향을 돌려 연산자를 바꾸므로 **쓰면 손해**다.

`--sigma-ablation` 은 가중치를 그대로 두고 σ 입력만 바꿔 σ 를 실제로 쓰는지 잰다 —
학습 없는 ablation 이라 대조군을 따로 60 에폭 돌릴 필요가 없다.

In [ ]:
import torch
rows = []
for ck in sorted(RUNS.glob("*/checkpoints/checkpoint_best.ckpt")):
    name = ck.parent.parent.name
    v = torch.load(ck, map_location="cpu", weights_only=False).get("val_psnr", -1)
    rows.append((v, name, ck))
    print(f"{name:<48}{v:>8.2f}")

for v, name, ck in sorted(rows, reverse=True):
    print(f"\n{'='*66}\n  {name}   (val {v:.2f})\n{'='*66}")
    !cd "{SRC}" && python eval_day3.py --data "{DATA_ROOT}" --ckpt "{ck}" --self-ensemble --sigma-ablation

## 7. 융합

구조가 다르면 틀리는 방식도 달라서 평균이 둘 다보다 좋다. 무게는 val 에서 고른다.
SSIM 도 같이 나오니, 융합으로 SSIM 이 떨어지면 쓰지 않으면 된다.

In [ ]:
CKS = [str(c) for _, _, c in sorted(rows, reverse=True)[:3]]
print("섞을 모델:", *[Path(c).parent.parent.name for c in CKS], sep="\n  ")
ARGS = " ".join(f'"{c}"' for c in CKS)
!cd "{SRC}" && python fuse_day3.py --data "{DATA_ROOT}" --ckpts {ARGS} --self-ensemble

## 8. 그림 — 노트북 안에서 바로 본다

배포 안내가 요구한 것: synthetic 학습 쌍과 test 결과 visualize, difference map,
detail zoom-in, 어떤 노이즈·이미지에 취약한지 분석.

In [ ]:
import json
from IPython.display import Image, display, Markdown

BEST = sorted(rows, reverse=True)[0][2]
cfg_p = BEST.parent.parent / "config.json"
cfg = json.loads(cfg_p.read_text(encoding="utf-8")) if cfg_p.exists() else {}
POST = "--post-wiener 0.00562" if cfg.get("target") == "measure" else ""
FIGDIR = Path("/content/figures"); FIGDIR.mkdir(exist_ok=True)

!cd "{SRC}" && python figures_day3.py --data "{DATA_ROOT}" --ckpt "{BEST}" {POST} --self-ensemble --out "{FIGDIR}"

TITLES = {
    "day3_forward_chain": "1. 열화 사슬 — 노이즈가 흐림 뒤에 붙는다",
    "day3_methods_grid":  "2. 노이즈 종류별 x 방법별 복원 결과",
    "day3_diff_zoom":     "3. difference map 과 zoom-in",
    "day3_weakness":      "4. 어떤 노이즈·어떤 σ 에 취약한가",
}
for stem, t in TITLES.items():
    p = FIGDIR / f"{stem}.png"
    if p.exists():
        display(Markdown(f"### {t}")); display(Image(filename=str(p), width=1100))

## 9. Drive 에 저장

체크포인트는 런타임이 끊기면 사라진다. 발표용 그림도 같이 내보낸다.

In [ ]:
import shutil
OUT = Path("/content/drive/MyDrive/ds_day3"); OUT.mkdir(parents=True, exist_ok=True)
for _, name, ck in rows:
    shutil.copy(ck, OUT / f"{name}.ckpt")
    cfg = ck.parent.parent / "config.json"
    if cfg.exists():
        shutil.copy(cfg, OUT / f"{name}_config.json")
for p in FIGDIR.glob("day3_*"):
    shutil.copy(p, OUT / p.name)
print("저장 ->", OUT)
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<52}{f.stat().st_size/1e6:>8.1f} MB")